In [6]:
import json
import re

input_file = "question_bank_final.jsonl"
output_file = "question_bank_final_cleaned.jsonl"

# fields where we just want to convert inline LaTeX
fields_to_update = [
    "question", "reasoning_steps", "paragraph_reasoning", "gt_solution", "Analysis"
]

# simple replace of inline LaTeX delimiters
def convert_inline_latex(text):
    if not isinstance(text, str):
        return text
    return text.replace("\\(", "$").replace("\\)", "$") \
               .replace("\\[", "$").replace("\\]", "$")

# build a paragraph from numbered steps
def steps_to_paragraph(steps_text):
    # split on "Step <number>:" (case‐insensitive)
    parts = re.split(r"(?i)step\s*\d+:\s*", steps_text)
    # the first element in parts may be "" if text starts with "Step 1:"
    cleaned = [p.strip() for p in parts if p.strip()]
    if not cleaned:
        return ""
    # rotating list of conjunctions
    connectors = [
        "Then",            # next
        "For the next step", 
        "Next",
        "Finally"
    ]
    # start with the first chunk as-is
    paragraph = cleaned[0].rstrip(".") + "."
    # append each subsequent chunk with a connector
    for i, chunk in enumerate(cleaned[1:], start=1):
        conj = connectors[(i-1) % len(connectors)]
        # ensure it ends in a period
        sentence = chunk.rstrip(".") + "."
        paragraph += f" {conj}, {sentence}"
    return paragraph

with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in infile:
        data = json.loads(line)

        # 1) first, clean inline LaTeX everywhere


        # 2) if there's reasoning_steps but no paragraph_reasoning, build one
        rs = data.get("reasoning_steps")
        pr = data.get("paragraph_reasoning")
        if isinstance(rs, str) and (not pr or not pr.strip()):
            data["paragraph_reasoning"] = steps_to_paragraph(rs)
            
        for field in fields_to_update:
            if field in data and isinstance(data[field], str):
                data[field] = convert_inline_latex(data[field])

        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")